In [ ]:
import random
import json
import re
from tqdm import tqdm

# --- 1. CONFIGURATION ---
TOTAL_SAMPLES = 50000
output_file = "datasetgeneree.jsonl"

SHARED_SYSTEM_PROMPT = (
    "You are a professional and friendly assistant. Your role is to help clients "
    "book, cancel, or reschedule appointments with precision. Always ensure you "
    "verify the service type, the day, and the specific time before confirming."
)

# --- 2. DICTIONNAIRES ENRICHIS (Contextual Data) ---
SERVICES = ["haircut", "dental checkup", "yoga class", "car maintenance", "consultation", "massage"]
LOCATIONS = ["Downtown clinic", "City Center branch", "West Plaza office", "North Branch"]
NAMES = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis"]
DAYS = ["Monday", "next Tuesday", "Wednesday", "tomorrow", "Friday", "April 10th"]
TIMES = ["9:00 AM", "10:30 AM", "2:00 PM", "4:45 PM", "11:15 AM", "5:30 PM"]

# Phrases contextuelles pour rendre le dialogue humain

#Contexte avant 
USER_STARTERS = [
    "Hi, I'm calling to book a {s}.",
    "Hello! I need a {s} appointment please.",
    "Hey there, can I schedule a {s}?",
    "I'd like to make an appointment for a {s}."
]

NAME_PROMPTS = [
    "Of course. Who should I put this reservation under?",
    "Sure thing. May I have your last name for the file?",
    "Perfect. And your name is?",
    "To finalize this, could you give me your name please?"
]

TIME_PROMPTS = [
    "I have two slots available: {t1} or {t2}. Which one do you prefer?",
    "For {d}, we can do {t1} or {t2}. What works best for you?",
    "Let me see... we have openings at {t1} and {t2}. Any preference?"
]

# --- 3. LOGIQUE DE GÉNÉRATION DE SCÉNARIO ---
def generate_human_scenario(intent):
    s = random.choice(SERVICES)
    d = random.choice(DAYS)
    n = random.choice(NAMES)
    loc = random.choice(LOCATIONS)
    t1, t2 = random.sample(TIMES, 2)
    health_id = f"ID-{random.randint(10000, 99999)}"

    flow = []
    flow.append({"role": "assistant", "content": "Hello! How can I help you today?"})

    if intent == "booking":
        # Tour 1: User request
        flow.append({"role": "user", "content": random.choice(USER_STARTERS).format(s=s)})

        # Tour 2: Assistant asks for Day
        flow.append({"role": "assistant", "content": f"I can certainly help you with that {s}. What day were you thinking of?"})
        flow.append({"role": "user", "content": f"I was hoping for {d}."})
#confirmation : contexte apres 

        # Tour 3: Assistant proposes times (Context before data)
        flow.append({"role": "assistant", "content": random.choice(TIME_PROMPTS).format(d=d, t1=t1, t2=t2)})
        flow.append({"role": "user", "content": f"Let's go with the {t1} slot, please."})

        # Tour 4: Ask for name (Contextual)
        flow.append({"role": "assistant", "content": random.choice(NAME_PROMPTS)})
        flow.append({"role": "user", "content": f"My name is {n}, and my health ID is {health_id}."})

        # Tour 5: Final Confirmation (Context after data)
        flow.append({"role": "assistant", "content": f"All set! Your {s} is confirmed at our {loc} for {d} at {t1}. We look forward to seeing you, {n}."})

    elif intent == "rescheduling":
        flow.append({"role": "user", "content": f"Hi, I have a {s} on {d} but I need to change it."})
        flow.append({"role": "assistant", "content": f"No problem at all. I can move your {s}. I have {t1} or {t2} available. Which do you prefer?"})
        flow.append({"role": "user", "content": f"Can we do {t1}?"})
        flow.append({"role": "assistant", "content": f"Perfect. I've updated your appointment at {loc}. It's now set for {t1} on {d}. Anything else?"})
        flow.append({"role": "user", "content": "No, that's all. Thanks!"})

    elif intent == "cancellation":
        flow.append({"role": "user", "content": f"I'm sorry, but I need to cancel my {s} for {d}."})
        flow.append({"role": "assistant", "content": f"I understand. I've removed your {s} for {d} from our schedule at {loc}. We hope to see you again soon."})
        flow.append({"role": "user", "content": "Thank you, goodbye."})

    else: # information
        flow.append({"role": "user", "content": f"Where is your {loc} office located exactly?"})
        flow.append({"role": "assistant", "content": f"Our {loc} is situated right next to the main square, near the station. We are open until {t2}."})
        flow.append({"role": "user", "content": "That's very helpful, thanks!"})

    return flow

# --- 4. BOUCLE DE GÉNÉRATION ---
synthetic_dataset = []
INTENTS = ["booking", "cancellation", "rescheduling", "information"]
samples_per_intent = TOTAL_SAMPLES // len(INTENTS)

print(f"Starting generation of {TOTAL_SAMPLES} samples...")

for intent in INTENTS:
    for _ in tqdm(range(samples_per_intent), desc=f"Generating {intent}"):
        conv = generate_human_scenario(intent)
        synthetic_dataset.append({
            "messages": [{"role": "system", "content": SHARED_SYSTEM_PROMPT}] + conv
        })

# Mélange final
random.shuffle(synthetic_dataset)

# --- 5. SAUVEGARDE ---
with open(output_file, "w", encoding="utf-8") as f:
    for entry in synthetic_dataset:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"\n✅ DONE! {len(synthetic_dataset)} human-like conversations generated in {output_file}")

Starting generation of 50000 samples...


Generating information: 100%|██████████| 12500/12500 [00:00<00:00, 121592.36it/s]



✅ DONE! 50000 human-like conversations generated in datasetgeneree.jsonl


In [2]:
'''
Phrases d'introduction (Context Before) : J'ai ajouté USER_STARTERS et NAME_PROMPTS pour que
l'IA n'entende pas juste "Booking Haircut", mais des phrases comme "Hi, I'm calling to book a...".
Structure de réponse (Context After) : Les confirmations finales répètent les données dans une phrase complète
 : "Your {s} is confirmed at {loc} for {d}...", ce qui aide le modèle à lier les entités entre elles.
Données mixtes : J'ai intégré le health_id et le name dans la même phrase pour apprendre au modèle à gérer
plusieurs informations d'un coup.
Variété : Le script propose maintenant deux créneaux horaires (t1 et t2) au lieu d'un seul,
pour simuler une vraie négociation de prise de rendez-vous.

'''

'\nPhrases d\'introduction (Context Before) : J\'ai ajouté USER_STARTERS et NAME_PROMPTS pour que\nl\'IA n\'entende pas juste "Booking Haircut", mais des phrases comme "Hi, I\'m calling to book a...".\nStructure de réponse (Context After) : Les confirmations finales répètent les données dans une phrase complète\n : "Your {s} is confirmed at {loc} for {d}...", ce qui aide le modèle à lier les entités entre elles.\nDonnées mixtes : J\'ai intégré le health_id et le name dans la même phrase pour apprendre au modèle à gérer\nplusieurs informations d\'un coup.\nVariété : Le script propose maintenant deux créneaux horaires (t1 et t2) au lieu d\'un seul,\npour simuler une vraie négociation de prise de rendez-vous.\n\n'